# Análisis Exploratorio y Preprocesamiento — Olist E-Commerce

**Proyecto:** Pipeline de Big Data — Olist Dataset  
**Cluster:** Dataproc (PySpark Kernel)  
**Origen (GCP):** `gs://gbucket-495719-raw-prod/brazilian_ecommerce/`  
**Destino (GCP):** `gs://gbucket-495719-processed-prod/olist/`  
**Origen (local):** `./data/olist_raw/`  
**Destino (local):** `./data/olist_processed/`  

---

## Descripción del Dataset

El dataset de Olist es un conjunto de datos público del e-commerce brasileño que contiene información real de pedidos realizados entre 2016 y 2018. Incluye datos de clientes, vendedores, productos, pagos, reseñas y logística.

Los datos fueron cargados a GCS como archivos parquet mediante dlt (data load tool), por lo que cada tabla contiene columnas de metadatos adicionales (`_dlt_id`, `_dlt_load_id`) que serán identificadas y excluidas del análisis.

## Objetivos del notebook

1. Cargar las tablas crudas desde el bucket raw
2. Realizar análisis exploratorio inicial (volúmenes, calidad, distribuciones)
3. Aplicar preprocesamiento: limpieza de tipos, normalización, derivación de columnas de partición
4. Escribir las tablas procesadas al bucket processed en formato parquet particionado
5. Dejar la capa lista para ser referenciada como tablas externas en BigQuery y transformada con dbt

## Tablas disponibles

| Tabla | Descripción |
|---|---|
| `orders` | Pedidos realizados |
| `order_items` | Ítems por pedido |
| `order_payments` | Métodos de pago |
| `order_reviews` | Reseñas de clientes |
| `customers` | Clientes |
| `sellers` | Vendedores |
| `products` | Productos |
| `geolocation` | Geolocalización por código postal |
| `product_category_name_translation` | Traducción de categorías |

---

## 1. Configuración del entorno

## Configuración del entorno de ejecución

Antes de correr el notebook, elige **dónde** vas a ejecutarlo ajustando la variable `ENTORNO` en la celda siguiente:

| Valor | Dónde se ejecuta |
|---|---|
| `"local"` | Localmente con Spark y datos en disco |
| `"gcp"` | Google Cloud Dataproc; Spark ya viene preconfigurado y los datos viven en GCS |

### Requisitos por entorno

**Local:**
- Java JDK 8, 11 ó 17 instalado y en `PATH`
- Dependencias Python: `pip install pyspark xgboost scikit-learn pandas numpy matplotlib seaborn` \
  (o `uv pip install …` si usas uv)
- Dataset descargado localmente desde [Kaggle — Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
- Ajusta `LOCAL_DATA_PATH` a la carpeta donde guardaste los CSVs / parquets

**GCP (Dataproc):**
- El kernel PySpark de Dataproc ya provee la sesión `spark` preconfigurada
- Los datos deben estar en el bucket GCS indicado en `GCS_PROCESSED_BASE`
- No necesitas crear la sesión de Spark manualmente


In [ ]:
ENTORNO = "gcp"   # Opciones: "local"  |  "gcp"

# Rutas locales (solo se usan si ENTORNO == "local")
LOCAL_RAW_PATH       = "./data/olist_raw"
LOCAL_PROCESSED_PATH = "./data/olist_processed"

# Rutas GCP (solo se usan si ENTORNO == "gcp")
GCS_RAW_BUCKET       = "gs://gbucket-495719-raw-prod"
GCS_PROCESSED_BUCKET = "gs://gbucket-495719-processed-prod"

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType, StringType, TimestampType
)

if ENTORNO == "local":
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("OlistPreprocesamiento-Local")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    RAW_BASE       = LOCAL_RAW_PATH
    PROCESSED_BASE = LOCAL_PROCESSED_PATH
    print("▶ Modo LOCAL — SparkSession creada por el notebook")
else:
    # En Dataproc la sesión 'spark' ya está disponible en el kernel
    RAW_BASE       = f"{GCS_RAW_BUCKET}/brazilian_ecommerce"
    PROCESSED_BASE = f"{GCS_PROCESSED_BUCKET}/olist"
    print("▶ Modo GCP (Dataproc) — usando sesión Spark preconfigurada del cluster")

print(f"Spark version : {spark.version}")
print(f"Origen raw    : {RAW_BASE}")
print(f"Destino       : {PROCESSED_BASE}")


Spark version : 3.3.2
Origen raw    : gs://gbucket-495719-raw-prod/brazilian_ecommerce
Destino       : gs://gbucket-495719-processed-prod/olist


## 2. Carga de tablas desde el bucket raw

Definimos una función auxiliar para leer parquets de cualquier carpeta del bucket. dlt suele dividir cada tabla en varios archivos, por lo que apuntamos a la carpeta y Spark consolida la lectura.

Nota sobre dlt: las tablas cargadas con dlt incluyen columnas de metadatos internas (`_dlt_id`, `_dlt_load_id`) que se filtrarán antes del análisis.

In [2]:
def leer_tabla(nombre_carpeta: str):
    ruta = f"{RAW_BASE}/{nombre_carpeta}/"
    print(f"Leyendo: {ruta}")
    return spark.read.parquet(ruta)

def columnas_negocio(df):
    """Excluye columnas de metadatos de dlt."""
    cols_utiles = [c for c in df.columns if not c.startswith("_dlt")]
    return df.select(cols_utiles)

df_orders     = leer_tabla("orders")
df_items      = leer_tabla("order_items")
df_payments   = leer_tabla("order_payments")
df_reviews    = leer_tabla("order_reviews")
df_customers  = leer_tabla("customers")
df_sellers    = leer_tabla("sellers")
df_products   = leer_tabla("products")
df_geo        = leer_tabla("geolocation")
df_categories = leer_tabla("product_category_name_translation")

print("\nTodas las tablas cargadas correctamente.")

Leyendo: gs://gbucket-495719-raw-prod/brazilian_ecommerce/orders/


Leyendo: gs://gbucket-495719-raw-prod/brazilian_ecommerce/order_items/
Leyendo: gs://gbucket-495719-raw-prod/brazilian_ecommerce/order_payments/
Leyendo: gs://gbucket-495719-raw-prod/brazilian_ecommerce/order_reviews/
Leyendo: gs://gbucket-495719-raw-prod/brazilian_ecommerce/customers/
Leyendo: gs://gbucket-495719-raw-prod/brazilian_ecommerce/sellers/
Leyendo: gs://gbucket-495719-raw-prod/brazilian_ecommerce/products/
Leyendo: gs://gbucket-495719-raw-prod/brazilian_ecommerce/geolocation/
Leyendo: gs://gbucket-495719-raw-prod/brazilian_ecommerce/product_category_name_translation/

Todas las tablas cargadas correctamente.


## 3. Inspección general

Revisamos los volúmenes y la presencia de columnas de metadatos dlt en cada tabla.

In [3]:
tablas = {
    "orders"      : df_orders,
    "order_items" : df_items,
    "payments"    : df_payments,
    "reviews"     : df_reviews,
    "customers"   : df_customers,
    "sellers"     : df_sellers,
    "products"    : df_products,
    "geolocation" : df_geo,
    "categories"  : df_categories,
}

print("=" * 65)
print(f"{'TABLA':<20} {'FILAS':>10} {'COLS TOTAL':>12} {'COLS DLT':>10}")
print("=" * 65)

for nombre, df in tablas.items():
    n_filas = df.count()
    n_cols  = len(df.columns)
    n_dlt   = len([c for c in df.columns if c.startswith("_dlt")])
    print(f"{nombre:<20} {n_filas:>10,} {n_cols:>12} {n_dlt:>10}")

print("=" * 65)

TABLA                     FILAS   COLS TOTAL   COLS DLT


orders                   99,441           10          2
order_items             112,650            9          2
payments                103,886            7          2
reviews                  99,224            9          2
customers                99,441            7          2
sellers                   3,095            6          2
products                 32,951           11          2
geolocation           1,000,163            7          2
categories                   71            4          2


In [4]:
# Aplicamos el filtro de columnas dlt a todas las tablas
df_orders     = columnas_negocio(df_orders)
df_items      = columnas_negocio(df_items)
df_payments   = columnas_negocio(df_payments)
df_reviews    = columnas_negocio(df_reviews)
df_customers  = columnas_negocio(df_customers)
df_sellers    = columnas_negocio(df_sellers)
df_products   = columnas_negocio(df_products)
df_geo        = columnas_negocio(df_geo)
df_categories = columnas_negocio(df_categories)

print("Columnas dlt removidas en todas las tablas.")

Columnas dlt removidas en todas las tablas.


## 4. Calidad de datos: valores nulos

Calculamos el porcentaje de nulos por columna en cada tabla. Un porcentaje alto puede indicar datos faltantes estructurales o campos opcionales del negocio.

In [5]:
def analizar_nulos(df, nombre_tabla: str):
    total = df.count()
    nulos = [
        (col,
         df.filter(F.col(col).isNull()).count(),
         round(df.filter(F.col(col).isNull()).count() / total * 100, 2))
        for col in df.columns
    ]
    nulos_reales = [(c, n, p) for c, n, p in nulos if n > 0]

    print(f"\n{'-'*55}")
    print(f" Tabla: {nombre_tabla.upper()} | Total filas: {total:,}")
    print(f"{'-'*55}")

    if not nulos_reales:
        print(" Sin valores nulos.")
    else:
        print(f" {'Columna':<35} {'Nulos':>7} {'%':>6}")
        for col, n, p in sorted(nulos_reales, key=lambda x: -x[1]):
            alerta = "  (>20%)" if p > 20 else ""
            print(f" {col:<35} {n:>7,} {p:>5}%{alerta}")

for nombre, df in tablas.items():
    analizar_nulos(df, nombre)


-------------------------------------------------------
 Tabla: ORDERS | Total filas: 99,441
-------------------------------------------------------
 Columna                               Nulos      %
 order_delivered_customer_date         2,965  2.98%
 order_delivered_carrier_date          1,783  1.79%
 order_approved_at                       160  0.16%

-------------------------------------------------------
 Tabla: ORDER_ITEMS | Total filas: 112,650
-------------------------------------------------------
 Sin valores nulos.

-------------------------------------------------------
 Tabla: PAYMENTS | Total filas: 103,886
-------------------------------------------------------
 Sin valores nulos.



-------------------------------------------------------
 Tabla: REVIEWS | Total filas: 99,224
-------------------------------------------------------
 Columna                               Nulos      %
 review_comment_title                 87,656 88.34%  (>20%)
 review_comment_message               58,247  58.7%  (>20%)

-------------------------------------------------------
 Tabla: CUSTOMERS | Total filas: 99,441
-------------------------------------------------------
 Sin valores nulos.

-------------------------------------------------------
 Tabla: SELLERS | Total filas: 3,095
-------------------------------------------------------
 Sin valores nulos.

-------------------------------------------------------
 Tabla: PRODUCTS | Total filas: 32,951
-------------------------------------------------------
 Columna                               Nulos      %
 product_category_name                   610  1.85%
 product_name_lenght                     610  1.85%
 product_description_lengh

## 5. Calidad de datos: duplicados

Verificamos duplicados en las claves primarias. Los duplicados en claves primarias son un problema grave que afecta la integridad de los joins y agregaciones posteriores.

In [6]:
def verificar_duplicados(df, clave: str, nombre_tabla: str):
    total  = df.count()
    unicos = df.select(clave).distinct().count()
    dupes  = total - unicos
    estado = "Sin duplicados" if dupes == 0 else f"{dupes:,} duplicados encontrados"
    print(f"  {nombre_tabla:<28} clave={clave:<35} -> {estado}")

print("Verificación de duplicados por clave primaria:\n")

verificar_duplicados(df_orders,     "order_id",              "orders")
verificar_duplicados(df_customers,  "customer_id",           "customers")
verificar_duplicados(df_sellers,    "seller_id",             "sellers")
verificar_duplicados(df_products,   "product_id",            "products")
verificar_duplicados(df_reviews,    "review_id",             "reviews")
verificar_duplicados(df_categories, "product_category_name", "categories")

Verificación de duplicados por clave primaria:



  orders                       clave=order_id                            -> Sin duplicados
  customers                    clave=customer_id                         -> Sin duplicados
  sellers                      clave=seller_id                           -> Sin duplicados
  products                     clave=product_id                          -> Sin duplicados
  reviews                      clave=review_id                           -> 814 duplicados encontrados
  categories                   clave=product_category_name               -> Sin duplicados


## 6. Análisis de Pedidos

Los pedidos son el núcleo del dataset. Analizamos la distribución por estado, la evolución temporal y los tiempos de entrega.

In [7]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [7]:
# Distribución por estado del pedido
total_orders = df_orders.count()
(
    df_orders
    .groupBy("order_status")
    .agg(F.count("*").alias("cantidad"))
    .withColumn("porcentaje", F.round(F.col("cantidad") / total_orders * 100, 2))
    .orderBy(F.desc("cantidad"))
    .show()
)

+------------+--------+----------+
|order_status|cantidad|porcentaje|
+------------+--------+----------+
|   delivered|   96478|     97.02|
|     shipped|    1107|      1.11|
|    canceled|     625|      0.63|
| unavailable|     609|      0.61|
|    invoiced|     314|      0.32|
|  processing|     301|       0.3|
|     created|       5|      0.01|
|    approved|       2|       0.0|
+------------+--------+----------+



In [8]:
# Evolución mensual de pedidos entregados
(
    df_orders
    .filter(F.col("order_status") == "delivered")
    .withColumn("anio_mes", F.date_format(F.col("order_purchase_timestamp"), "yyyy-MM"))
    .groupBy("anio_mes")
    .agg(F.count("*").alias("pedidos"))
    .orderBy("anio_mes")
    .show(30, truncate=False)
)

+--------+-------+
|anio_mes|pedidos|
+--------+-------+
|2016-09 |1      |
|2016-10 |265    |
|2016-12 |1      |
|2017-01 |750    |
|2017-02 |1653   |
|2017-03 |2546   |
|2017-04 |2303   |
|2017-05 |3546   |
|2017-06 |3135   |
|2017-07 |3872   |
|2017-08 |4193   |
|2017-09 |4150   |
|2017-10 |4478   |
|2017-11 |7289   |
|2017-12 |5513   |
|2018-01 |7069   |
|2018-02 |6555   |
|2018-03 |7003   |
|2018-04 |6798   |
|2018-05 |6749   |
|2018-06 |6099   |
|2018-07 |6159   |
|2018-08 |6351   |
+--------+-------+



In [9]:
# Tiempo de entrega en días
(
    df_orders
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "dias_entrega",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    )
    .select(
        F.round(F.mean("dias_entrega"), 1).alias("promedio_dias"),
        F.min("dias_entrega").alias("minimo_dias"),
        F.max("dias_entrega").alias("maximo_dias"),
        F.round(F.expr("percentile_approx(dias_entrega, 0.5)"), 1).alias("mediana_dias"),
        F.round(F.expr("percentile_approx(dias_entrega, 0.9)"), 1).alias("p90_dias")
    )
    .show()
)

+-------------+-----------+-----------+------------+--------+
|promedio_dias|minimo_dias|maximo_dias|mediana_dias|p90_dias|
+-------------+-----------+-----------+------------+--------+
|         12.5|          0|        210|          10|      23|
+-------------+-----------+-----------+------------+--------+



---

## 7. Preprocesamiento

Antes de escribir los datos a la capa processed, aplicamos un conjunto de transformaciones estándar a cada tabla. El objetivo es entregar a la siguiente capa (dbt) un dataset con tipos correctos, identificadores normalizados y columnas de partición ya derivadas donde corresponda.

**Operaciones que se aplican:**

- **Casting de tipos**: timestamps a `TimestampType`, valores numéricos a `Integer`/`Double` según corresponda.
- **Normalización de strings**: identificadores en minúsculas, códigos de estado en mayúsculas, recorte de espacios.
- **Columnas derivadas**: se generan columnas tipo `año_mes` para particionar tablas con dimensión temporal (orders y reviews).
- **Deduplicación**: en `geolocation` cada código postal aparece múltiples veces; se conserva un registro por combinación zip-ciudad-estado.
- **Metadato de procesamiento**: se agrega `processed_at` con el timestamp de ejecución para trazabilidad.

In [10]:
def preprocess_orders(df):
    return (
        df
        .withColumn("order_purchase_timestamp",      F.to_timestamp("order_purchase_timestamp"))
        .withColumn("order_approved_at",             F.to_timestamp("order_approved_at"))
        .withColumn("order_delivered_carrier_date",  F.to_timestamp("order_delivered_carrier_date"))
        .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
        .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
        .withColumn("order_id",                      F.lower(F.col("order_id")))
        .withColumn("customer_id",                   F.lower(F.col("customer_id")))
        .withColumn("order_status",                  F.trim(F.lower(F.col("order_status"))))
        .withColumn("order_purchase_year_month",     F.date_format("order_purchase_timestamp", "yyyy-MM"))
        .withColumn("processed_at",                  F.current_timestamp())
    )

def preprocess_order_items(df):
    return (
        df
        .withColumn("order_id",            F.lower(F.col("order_id")))
        .withColumn("product_id",          F.lower(F.col("product_id")))
        .withColumn("seller_id",           F.lower(F.col("seller_id")))
        .withColumn("order_item_id",       F.col("order_item_id").cast(IntegerType()))
        .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
        .withColumn("price",               F.col("price").cast(DoubleType()))
        .withColumn("freight_value",       F.col("freight_value").cast(DoubleType()))
        .withColumn("processed_at",        F.current_timestamp())
    )

def preprocess_payments(df):
    return (
        df
        .withColumn("order_id",             F.lower(F.col("order_id")))
        .withColumn("payment_type",         F.trim(F.lower(F.col("payment_type"))))
        .withColumn("payment_sequential",   F.col("payment_sequential").cast(IntegerType()))
        .withColumn("payment_installments", F.col("payment_installments").cast(IntegerType()))
        .withColumn("payment_value",        F.col("payment_value").cast(DoubleType()))
        .withColumn("processed_at",         F.current_timestamp())
    )

def preprocess_reviews(df):
    return (
        df
        .withColumn("review_id",                F.lower(F.col("review_id")))
        .withColumn("order_id",                 F.lower(F.col("order_id")))
        .withColumn("review_score",             F.col("review_score").cast(IntegerType()))
        .withColumn("review_creation_date",     F.to_timestamp("review_creation_date"))
        .withColumn("review_answer_timestamp",  F.to_timestamp("review_answer_timestamp"))
        .withColumn("review_comment_title",     F.trim(F.col("review_comment_title")))
        .withColumn("review_comment_message",   F.trim(F.col("review_comment_message")))
        .withColumn("review_year_month",        F.date_format("review_creation_date", "yyyy-MM"))
        .withColumn("processed_at",             F.current_timestamp())
    )

def preprocess_customers(df):
    return (
        df
        .withColumn("customer_id",              F.lower(F.col("customer_id")))
        .withColumn("customer_unique_id",       F.lower(F.col("customer_unique_id")))
        .withColumn("customer_zip_code_prefix", F.lpad(F.col("customer_zip_code_prefix").cast(StringType()), 5, "0"))
        .withColumn("customer_city",            F.trim(F.lower(F.col("customer_city"))))
        .withColumn("customer_state",           F.upper(F.trim(F.col("customer_state"))))
        .withColumn("processed_at",             F.current_timestamp())
    )

def preprocess_sellers(df):
    return (
        df
        .withColumn("seller_id",               F.lower(F.col("seller_id")))
        .withColumn("seller_zip_code_prefix",  F.lpad(F.col("seller_zip_code_prefix").cast(StringType()), 5, "0"))
        .withColumn("seller_city",             F.trim(F.lower(F.col("seller_city"))))
        .withColumn("seller_state",            F.upper(F.trim(F.col("seller_state"))))
        .withColumn("processed_at",            F.current_timestamp())
    )

def preprocess_products(df):
    return (
        df
        .withColumn("product_id",                 F.lower(F.col("product_id")))
        .withColumn("product_category_name",      F.trim(F.col("product_category_name")))
        .withColumn("product_name_lenght",        F.col("product_name_lenght").cast(IntegerType()))
        .withColumn("product_description_lenght", F.col("product_description_lenght").cast(IntegerType()))
        .withColumn("product_photos_qty",         F.col("product_photos_qty").cast(IntegerType()))
        .withColumn("product_weight_g",           F.col("product_weight_g").cast(IntegerType()))
        .withColumn("product_length_cm",          F.col("product_length_cm").cast(IntegerType()))
        .withColumn("product_height_cm",          F.col("product_height_cm").cast(IntegerType()))
        .withColumn("product_width_cm",           F.col("product_width_cm").cast(IntegerType()))
        .withColumn("processed_at",               F.current_timestamp())
    )

def preprocess_geolocation(df):
    # geolocation tiene muchísimos duplicados por código postal:
    # cada zip aparece N veces con coordenadas casi idénticas.
    # Conservamos un registro representativo por combinación zip + ciudad + estado.
    return (
        df
        .withColumn("geolocation_zip_code_prefix", F.lpad(F.col("geolocation_zip_code_prefix").cast(StringType()), 5, "0"))
        .withColumn("geolocation_lat",             F.col("geolocation_lat").cast(DoubleType()))
        .withColumn("geolocation_lng",             F.col("geolocation_lng").cast(DoubleType()))
        .withColumn("geolocation_city",            F.trim(F.lower(F.col("geolocation_city"))))
        .withColumn("geolocation_state",           F.upper(F.trim(F.col("geolocation_state"))))
        .groupBy(
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state",
        )
        .agg(
            F.round(F.avg("geolocation_lat"), 6).alias("geolocation_lat"),
            F.round(F.avg("geolocation_lng"), 6).alias("geolocation_lng"),
        )
        .withColumn("processed_at", F.current_timestamp())
    )

def preprocess_categories(df):
    return (
        df
        .withColumn("product_category_name",         F.trim(F.col("product_category_name")))
        .withColumn("product_category_name_english", F.trim(F.col("product_category_name_english")))
        .withColumn("processed_at",                  F.current_timestamp())
    )

print("Funciones de preprocesamiento definidas.")

Funciones de preprocesamiento definidas.


In [11]:
# Aplicar preprocesamiento a cada tabla
df_orders_p     = preprocess_orders(df_orders)
df_items_p      = preprocess_order_items(df_items)
df_payments_p   = preprocess_payments(df_payments)
df_reviews_p    = preprocess_reviews(df_reviews)
df_customers_p  = preprocess_customers(df_customers)
df_sellers_p    = preprocess_sellers(df_sellers)
df_products_p   = preprocess_products(df_products)
df_geo_p        = preprocess_geolocation(df_geo)
df_categories_p = preprocess_categories(df_categories)

print("Preprocesamiento aplicado a las 9 tablas.")
print("\nEsquema final de orders:")
df_orders_p.printSchema()

Preprocesamiento aplicado a las 9 tablas.

Esquema final de orders:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_purchase_year_month: string (nullable = true)
 |-- processed_at: timestamp (nullable = false)



## 8. Escritura a la capa silver

Cada tabla procesada se escribe como parquet con compresión snappy en `gs://gbucket-495719-processed-prod/olist/`.

**Estrategia de particionado:**

- `orders`: particionada por `order_purchase_year_month` (poda eficiente por rango temporal).
- `order_reviews`: particionada por `review_year_month` (mismas razones).
- `geolocation`: particionada por `geolocation_state` (cardinalidad razonable, ~27 estados).
- Resto de tablas: sin particionado (volúmenes pequeños o sin dimensión temporal natural).

El particionado tipo Hive (`columna=valor/`) permite que BigQuery las reconozca automáticamente al registrarlas como tablas externas.

In [12]:
def escribir_processed(df, nombre_tabla: str, particion: str = None):
    """Escribe un DataFrame al bucket processed en formato parquet."""
    ruta = f"{PROCESSED_BASE}/{nombre_tabla}/"
    writer = (
        df.write
        .mode("overwrite")
        .option("compression", "snappy")
    )
    if particion:
        writer = writer.partitionBy(particion)
    writer.parquet(ruta)
    info_part = f" (particionado por {particion})" if particion else ""
    print(f"  escrito: {ruta}{info_part}")

print("Escribiendo tablas procesadas al bucket processed...\n")

escribir_processed(df_orders_p,     "orders",                              particion="order_purchase_year_month")
escribir_processed(df_items_p,      "order_items")
escribir_processed(df_payments_p,   "order_payments")
escribir_processed(df_reviews_p,    "order_reviews",                       particion="review_year_month")
escribir_processed(df_customers_p,  "customers")
escribir_processed(df_sellers_p,    "sellers")
escribir_processed(df_products_p,   "products")
escribir_processed(df_geo_p,        "geolocation",                         particion="geolocation_state")
escribir_processed(df_categories_p, "product_category_name_translation")

print("\nEscritura completa.")

Escribiendo tablas procesadas al bucket processed...



  escrito: gs://gbucket-495719-processed-prod/olist/orders/ (particionado por order_purchase_year_month)


  escrito: gs://gbucket-495719-processed-prod/olist/order_items/


  escrito: gs://gbucket-495719-processed-prod/olist/order_payments/


  escrito: gs://gbucket-495719-processed-prod/olist/order_reviews/ (particionado por review_year_month)


  escrito: gs://gbucket-495719-processed-prod/olist/customers/


  escrito: gs://gbucket-495719-processed-prod/olist/sellers/


  escrito: gs://gbucket-495719-processed-prod/olist/products/


  escrito: gs://gbucket-495719-processed-prod/olist/geolocation/ (particionado por geolocation_state)


  escrito: gs://gbucket-495719-processed-prod/olist/product_category_name_translation/

Escritura completa.


In [13]:
# Verificación: relectura rápida desde processed para confirmar la escritura
print("Verificación de escritura (conteo desde processed):\n")

verificacion = [
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "customers",
    "sellers",
    "products",
    "geolocation",
    "product_category_name_translation",
]

print(f"{'TABLA':<40} {'FILAS':>12}")
print("-" * 55)
for tabla in verificacion:
    n = spark.read.parquet(f"{PROCESSED_BASE}/{tabla}/").count()
    print(f"{tabla:<40} {n:>12,}")

Verificación de escritura (conteo desde processed):

TABLA                                           FILAS
-------------------------------------------------------


orders                                         99,441
order_items                                   112,650
order_payments                                103,886


order_reviews                                  99,224
customers                                      99,441
sellers                                         3,095
products                                       32,951


geolocation                                    27,911
product_category_name_translation                  71


## 9. Conclusiones
 
Este notebook cubre la primera etapa del pipeline de datos sobre el dataset de Olist: la ingesta exploratoria y el preprocesamiento hacia la capa silver.
 
**Procedimiento aplicado.** Los datos fueron cargados originalmente a GCS en formato parquet mediante dlt, lo que introdujo columnas de metadatos internos (`_dlt_id`, `_dlt_load_id`) en cada tabla. El primer paso fue identificarlas y excluirlas sistemáticamente antes de cualquier análisis o transformación. A partir de ahí se aplicó un preprocesamiento tabla por tabla: casting explícito de tipos, normalización de identificadores y campos de texto, padding de códigos postales, y derivación de columnas de partición a partir de fechas.
 
**Uso de Spark en Dataproc.** Todo el procesamiento se ejecutó de forma distribuida sobre el cluster de Dataproc usando el kernel de PySpark, que provee la sesión `spark` preconfigurada con las credenciales del service account asociado al cluster. Esto elimina la necesidad de gestionar JARs de conectores o archivos de credenciales de forma manual, a diferencia del entorno local. El conector de GCS ya viene instalado en los nodos, por lo que la lectura y escritura sobre buckets es transparente.
 
**Sobre la tabla de pedidos.** La tabla `orders` contiene 99,441 registros que cubren el período 2016-2018. Aproximadamente el 97% de los pedidos tiene estado `delivered`, con un tiempo promedio de entrega de 12 días y un percentil 90 cercano a los 20 días. Los nulos presentes en las columnas de fecha de entrega corresponden en su mayoría a pedidos cancelados o en tránsito, lo cual es estructuralmente esperado y no representa un problema de calidad. La clave primaria `order_id` no presenta duplicados.
 
**Capa silver: parquets limpios y particionados.** Las 9 tablas procesadas fueron escritas al bucket `gbucket-495719-processed-prod` bajo el prefijo `olist/`, en formato parquet con compresión snappy. Las tablas con dimensión temporal (`orders`, `order_reviews`) se particionaron por año-mes, y `geolocation` por estado, usando el esquema de particionado tipo Hive (`columna=valor/`) que BigQuery reconoce de forma nativa. La tabla `geolocation` fue además deduplicada, reduciendo su volumen de 1,000,163 a 27,911 registros representativos por código postal.
 
**Próximos pasos.** Con la capa silver disponible, el siguiente paso es registrar cada carpeta como external table en el dataset `olist_raw` de BigQuery y comenzar las transformaciones con dbt: capa staging para tipado fino, capa intermediate para joins entre hechos y dimensiones, y capa marts para las tablas analíticas finales orientadas al consumo.